# 05 Monte Carlo Simulation

In this notebook I run the first probabilistic simulation of the cascaded cooling system. The aim is to move from the deterministic baseline into a set of repeated sampled operating states, so that overload risk can be estimated for HVAC, CW and FW.

Two uncertainty models are implemented and compared:

* triangular distribution;
* truncated normal distribution.

Both models are applied consistently across all subsystems, scenarios and uncertainty cases. I keep the sampled rows independent in this notebook, because dependency modelling is treated separately in Notebook 06.


## Method summary

The simulation uses the local load rows defined in Notebook 04. The embedded cascade rows are not sampled directly. Instead, each Monte Carlo iteration rebuilds the cascade from sampled local loads:

1. sample the local HVAC, CW and FW load rows;
2. sum the sampled HVAC local loads;
3. pass the sampled HVAC total into CW;
4. pass the sampled CW total into FW;
5. compare the resulting subsystem loads against the fixed capacity values from Notebook 03.

This preserves the physical cascade while avoiding double counting the workbook propagation rows.


## Imports and paths


In [ ]:
from pathlib import Path
import math
import time
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    plotting_available = True
except ModuleNotFoundError:
    plotting_available = False

try:
    from scipy.stats import truncnorm
    scipy_available = True
except ModuleNotFoundError:
    scipy_available = False

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (
            (candidate / "data" / "raw" / "data.xlsx").exists()
            and (candidate / "outputs" / "tables" / "uncertainty_distribution_parameters.csv").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find the project root. Please open VS Code at the repository folder.")


project_root = find_project_root()
tables_output_path = project_root / "outputs" / "tables"
figures_output_path = project_root / "outputs" / "figures"
tables_output_path.mkdir(parents=True, exist_ok=True)
figures_output_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Tables output path: {tables_output_path}")
print(f"Figures output path: {figures_output_path}")


## Load simulation inputs


In [ ]:
parameter_path = tables_output_path / "uncertainty_distribution_parameters.csv"
baseline_path = tables_output_path / "deterministic_baseline.csv"
excluded_rows_path = tables_output_path / "uncertainty_excluded_propagation_rows.csv"

distribution_parameters = pd.read_csv(parameter_path)
deterministic_baseline = pd.read_csv(baseline_path)
excluded_propagation_rows = pd.read_csv(excluded_rows_path)

system_order = ["HVAC", "CW", "FW"]
scenario_order = ["Scenario Alpha", "Scenario Bravo"]
margin_case_order = [
    "Case A - CUM",
    "Case B - CMM + CUM",
    "Case C - CMM + CUM + DBM + IGM",
]
distribution_order = ["Triangular", "Truncated normal"]

distribution_parameters["system"] = pd.Categorical(
    distribution_parameters["system"],
    categories=system_order,
    ordered=True,
)
distribution_parameters["scenario"] = pd.Categorical(
    distribution_parameters["scenario"],
    categories=scenario_order,
    ordered=True,
)
distribution_parameters["margin_case"] = pd.Categorical(
    distribution_parameters["margin_case"],
    categories=margin_case_order,
    ordered=True,
)

print(f"Distribution parameter rows: {len(distribution_parameters):,}")
print(f"Deterministic baseline rows: {len(deterministic_baseline):,}")
display(distribution_parameters.head())


## Monte Carlo notation


In [ ]:
notation_table = pd.DataFrame([
    {
        "Notation": "$N$",
        "Meaning": "Number of Monte Carlo iterations",
        "Use in this notebook": "Set to 100,000 for each scenario, margin case and uncertainty model."
    },
    {
        "Notation": "$i$",
        "Meaning": "Load item index",
        "Use in this notebook": "Each row in the uncertainty parameter table is sampled independently."
    },
    {
        "Notation": "$s$",
        "Meaning": "Subsystem index",
        "Use in this notebook": "HVAC, CW and FW."
    },
    {
        "Notation": "$j$",
        "Meaning": "Monte Carlo iteration index",
        "Use in this notebook": "One sampled operating state of the cascaded cooling system."
    },
    {
        "Notation": "$L_{i,j}$",
        "Meaning": "Sampled load for item $i$ in iteration $j$",
        "Use in this notebook": "Drawn from either the triangular or truncated normal model."
    },
    {
        "Notation": "$T_{s,j}$",
        "Meaning": "Sampled local total for subsystem $s$ in iteration $j$",
        "Use in this notebook": "Sum of local sampled item loads before cascade propagation."
    },
    {
        "Notation": "$Y_{s,j}$",
        "Meaning": "Cascaded subsystem load for subsystem $s$ in iteration $j$",
        "Use in this notebook": "The load compared against capacity after upstream loads are propagated."
    },
    {
        "Notation": "$C_s$",
        "Meaning": "Capacity of subsystem $s$",
        "Use in this notebook": "Fixed capacity from the deterministic baseline."
    },
    {
        "Notation": "$I(Y_{s,j} > C_s)$",
        "Meaning": "Overload indicator",
        "Use in this notebook": "Equals 1 when sampled load exceeds capacity, otherwise 0."
    },
    {
        "Notation": "$\\hat{P}_{f,s}$",
        "Meaning": "Estimated overload probability for subsystem $s$",
        "Use in this notebook": "Mean overload indicator across all $N$ iterations."
    },
])

notation_table.to_csv(tables_output_path / "monte_carlo_notation_table.csv", index=False)
display(notation_table)


## Simulation settings


<!-- monte-carlo-parameter-markdown-table -->

Table. Summary of Monte Carlo simulation parameters used in Notebook 05.

| Parameter | Notation | Value used | Reason / interpretation |
|---|---:|---|---|
| Number of simulations | $N$ | 100,000 | Provides a stable estimate of high-percentile load behaviour while remaining practical to run. |
| Random seed | - | 19046600 | Makes the simulation reproducible. |
| Operating scenarios | - | Scenario Alpha; Scenario Bravo | Both operating states from the workbook are assessed. |
| Uncertainty cases | - | Case A - CUM; Case B - CMM + CUM; Case C - CMM + CUM + DBM + IGM | Tests increasing levels of uncertainty included in the load model. |
| Uncertainty models | - | Triangular; truncated normal | Compares sensitivity to distributional assumptions. |
| Dependency assumption | - | Independent item-level sampling | Provides the baseline model before dependency is introduced in Notebook 06. |
| Cascade treatment | $Y_{s,j}$ | Recomputed each iteration from sampled local subsystem totals | Prevents double-counting of the embedded workbook propagation rows. |
| Capacity treatment | $C_s$ | Fixed deterministic capacity by subsystem and scenario | Compares sampled load uncertainty against the design capacity values. |
| Truncated normal standard deviation | $\sigma_i$ | $(u_i L_i) / 1.96$ | Treats the uncertainty band as an approximate 95% interval around nominal load. |
| Chunk size | - | 150 load rows | Limits memory use when sampling many rows across 100,000 iterations. |


In [ ]:
N_SIMULATIONS = 100_000
RANDOM_SEED = 19046600
CHUNK_SIZE = 150
PLOT_SAMPLE_SIZE = 2_000

simulation_settings = pd.DataFrame([
    {"Parameter": "Number of simulations", "Notation": "$N$", "Value": f"{N_SIMULATIONS:,}"},
    {"Parameter": "Random seed", "Notation": "-", "Value": RANDOM_SEED},
    {"Parameter": "Operating scenarios", "Notation": "-", "Value": ", ".join(scenario_order)},
    {"Parameter": "Uncertainty cases", "Notation": "-", "Value": "; ".join(margin_case_order)},
    {"Parameter": "Uncertainty models", "Notation": "-", "Value": ", ".join(distribution_order)},
    {"Parameter": "Dependency assumption", "Notation": "-", "Value": "Independent item-level sampling"},
    {"Parameter": "Cascade treatment", "Notation": "$Y_{s,j}$", "Value": "Recomputed each iteration from sampled local subsystem totals"},
    {"Parameter": "Capacity treatment", "Notation": "$C_s$", "Value": "Fixed deterministic capacity by subsystem and scenario"},
    {"Parameter": "Truncated normal standard deviation", "Notation": "$\\sigma_i$", "Value": "95% interval assumption: (uncertainty band x nominal load) / 1.96"},
    {"Parameter": "Chunk size", "Notation": "-", "Value": CHUNK_SIZE},
])

simulation_settings.to_csv(tables_output_path / "monte_carlo_simulation_settings.csv", index=False)
display(simulation_settings)


## Check the simulation design


In [ ]:
design_summary = (
    distribution_parameters
    .groupby(["scenario", "margin_case", "system"], observed=False)
    .agg(
        records=("record_id", "size"),
        non_degenerate_records=("is_degenerate", lambda values: int((~values).sum())),
        nominal_local_load_kw=("nominal_load_kw", "sum"),
    )
    .reset_index()
)

design_summary.to_csv(tables_output_path / "monte_carlo_design_summary.csv", index=False)
display(design_summary)


## Capacity lookup


In [ ]:
capacity_lookup = {
    (row["scenario"], row["subsystem"]): row["capacity_kw"]
    for _, row in deterministic_baseline.iterrows()
}

deterministic_load_lookup = {
    (row["scenario"], row["subsystem"]): row["deterministic_load_kw"]
    for _, row in deterministic_baseline.iterrows()
}

display(deterministic_baseline[[
    "scenario",
    "subsystem",
    "deterministic_load_kw",
    "capacity_kw",
    "capacity_margin_pct",
    "pass_fail",
]])


## Sampling functions


In [ ]:
def sample_variable_rows(row_chunk, distribution_model, n_simulations, rng):
    row_count = len(row_chunk)

    if row_count == 0:
        return np.empty((0, n_simulations))

    if distribution_model == "Triangular":
        lower = row_chunk["triangular_lower_kw"].to_numpy(dtype=float)[:, None]
        mode = row_chunk["triangular_mode_kw"].to_numpy(dtype=float)[:, None]
        upper = row_chunk["triangular_upper_kw"].to_numpy(dtype=float)[:, None]
        return rng.triangular(lower, mode, upper, size=(row_count, n_simulations))

    if distribution_model == "Truncated normal":
        if not scipy_available:
            raise ModuleNotFoundError("scipy is required for truncated normal sampling.")

        mean = row_chunk["truncnorm_mean_kw"].to_numpy(dtype=float)[:, None]
        lower = row_chunk["truncnorm_lower_kw"].to_numpy(dtype=float)[:, None]
        upper = row_chunk["truncnorm_upper_kw"].to_numpy(dtype=float)[:, None]
        sigma = row_chunk["truncnorm_sigma_kw"].to_numpy(dtype=float)[:, None]

        a = (lower - mean) / sigma
        b = (upper - mean) / sigma
        return truncnorm.rvs(
            a,
            b,
            loc=mean,
            scale=sigma,
            size=(row_count, n_simulations),
            random_state=rng,
        )

    raise ValueError(f"Unknown distribution model: {distribution_model}")


def sample_local_totals(parameters, distribution_model, n_simulations, rng, chunk_size):
    local_totals = {
        system: np.zeros(n_simulations, dtype=float)
        for system in system_order
    }

    for system in system_order:
        system_parameters = parameters[parameters["system"] == system].copy()

        fixed_parameters = system_parameters[system_parameters["is_degenerate"]]
        variable_parameters = system_parameters[~system_parameters["is_degenerate"]].reset_index(drop=True)

        local_totals[system] += fixed_parameters["nominal_load_kw"].sum()

        for start in range(0, len(variable_parameters), chunk_size):
            row_chunk = variable_parameters.iloc[start:start + chunk_size]
            sampled_rows = sample_variable_rows(
                row_chunk,
                distribution_model,
                n_simulations,
                rng,
            )
            local_totals[system] += sampled_rows.sum(axis=0)

    return local_totals


def rebuild_cascade(local_totals):
    hvac_total = local_totals["HVAC"]
    cw_total = local_totals["CW"] + hvac_total
    fw_total = local_totals["FW"] + cw_total

    return {
        "HVAC": hvac_total,
        "CW": cw_total,
        "FW": fw_total,
    }


def rule_of_three_upper_bound(overload_count, n_simulations):
    if overload_count == 0:
        return 3 / n_simulations
    return np.nan


def summarise_subsystem(values, scenario, margin_case, distribution_model, subsystem):
    capacity = capacity_lookup[(scenario, subsystem)]
    deterministic_load = deterministic_load_lookup[(scenario, subsystem)]
    overload = values > capacity
    overload_count = int(overload.sum())
    overload_probability = overload_count / len(values)
    standard_error = math.sqrt(overload_probability * (1 - overload_probability) / len(values))
    confidence_delta = 1.96 * standard_error

    return {
        "scenario": scenario,
        "margin_case": margin_case,
        "distribution_model": distribution_model,
        "subsystem": subsystem,
        "simulations": len(values),
        "deterministic_load_kw": deterministic_load,
        "capacity_kw": capacity,
        "mean_load_kw": values.mean(),
        "std_load_kw": values.std(ddof=1),
        "p01_load_kw": np.percentile(values, 1),
        "p05_load_kw": np.percentile(values, 5),
        "p50_load_kw": np.percentile(values, 50),
        "p95_load_kw": np.percentile(values, 95),
        "p99_load_kw": np.percentile(values, 99),
        "min_load_kw": values.min(),
        "max_load_kw": values.max(),
        "mean_spare_capacity_kw": capacity - values.mean(),
        "p95_spare_capacity_kw": capacity - np.percentile(values, 95),
        "p99_spare_capacity_kw": capacity - np.percentile(values, 99),
        "overload_count": overload_count,
        "overload_probability": overload_probability,
        "overload_probability_pct": overload_probability * 100,
        "overload_probability_ci95_lower": max(0, overload_probability - confidence_delta),
        "overload_probability_ci95_upper": min(1, overload_probability + confidence_delta),
        "rule_of_three_upper_if_zero": rule_of_three_upper_bound(overload_count, len(values)),
    }


## Run the Monte Carlo simulation


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
plot_sample_rows = []
run_log_rows = []

start_time = time.perf_counter()

for scenario in scenario_order:
    for margin_case in margin_case_order:
        combo_parameters = distribution_parameters[
            (distribution_parameters["scenario"] == scenario) &
            (distribution_parameters["margin_case"] == margin_case)
        ].copy()

        for distribution_model in distribution_order:
            run_start = time.perf_counter()
            local_totals = sample_local_totals(
                combo_parameters,
                distribution_model,
                N_SIMULATIONS,
                rng,
                CHUNK_SIZE,
            )
            cascaded_totals = rebuild_cascade(local_totals)

            for subsystem in system_order:
                values = cascaded_totals[subsystem]
                summary_rows.append(
                    summarise_subsystem(
                        values,
                        scenario,
                        margin_case,
                        distribution_model,
                        subsystem,
                    )
                )

                sample_size = min(PLOT_SAMPLE_SIZE, len(values))
                sample_indices = rng.choice(len(values), size=sample_size, replace=False)
                for value in values[sample_indices]:
                    plot_sample_rows.append({
                        "scenario": scenario,
                        "margin_case": margin_case,
                        "distribution_model": distribution_model,
                        "subsystem": subsystem,
                        "sampled_load_kw": value,
                    })

            run_log_rows.append({
                "scenario": scenario,
                "margin_case": margin_case,
                "distribution_model": distribution_model,
                "runtime_seconds": time.perf_counter() - run_start,
                "parameter_rows": len(combo_parameters),
                "non_degenerate_rows": int((~combo_parameters["is_degenerate"]).sum()),
            })

            print(
                f"Completed {scenario} | {margin_case} | {distribution_model} "
                f"in {run_log_rows[-1]['runtime_seconds']:.1f} seconds"
            )

total_runtime_seconds = time.perf_counter() - start_time
print(f"Total simulation runtime: {total_runtime_seconds:.1f} seconds")


## Simulation results


In [ ]:
monte_carlo_results = pd.DataFrame(summary_rows)
monte_carlo_plot_samples = pd.DataFrame(plot_sample_rows)
monte_carlo_run_log = pd.DataFrame(run_log_rows)

monte_carlo_results["scenario"] = pd.Categorical(
    monte_carlo_results["scenario"],
    categories=scenario_order,
    ordered=True,
)
monte_carlo_results["margin_case"] = pd.Categorical(
    monte_carlo_results["margin_case"],
    categories=margin_case_order,
    ordered=True,
)
monte_carlo_results["distribution_model"] = pd.Categorical(
    monte_carlo_results["distribution_model"],
    categories=distribution_order,
    ordered=True,
)
monte_carlo_results["subsystem"] = pd.Categorical(
    monte_carlo_results["subsystem"],
    categories=system_order,
    ordered=True,
)

monte_carlo_results["mean_load_to_capacity_ratio"] = (
    monte_carlo_results["mean_load_kw"] / monte_carlo_results["capacity_kw"]
)
monte_carlo_results["p95_load_to_capacity_ratio"] = (
    monte_carlo_results["p95_load_kw"] / monte_carlo_results["capacity_kw"]
)
monte_carlo_results["p99_load_to_capacity_ratio"] = (
    monte_carlo_results["p99_load_kw"] / monte_carlo_results["capacity_kw"]
)
monte_carlo_results["max_load_to_capacity_ratio"] = (
    monte_carlo_results["max_load_kw"] / monte_carlo_results["capacity_kw"]
)
monte_carlo_results["p95_capacity_margin_pct"] = (
    monte_carlo_results["p95_spare_capacity_kw"] / monte_carlo_results["capacity_kw"] * 100
)
monte_carlo_results["p99_capacity_margin_pct"] = (
    monte_carlo_results["p99_spare_capacity_kw"] / monte_carlo_results["capacity_kw"] * 100
)

monte_carlo_results = monte_carlo_results.sort_values([
    "scenario",
    "margin_case",
    "distribution_model",
    "subsystem",
]).reset_index(drop=True)

monte_carlo_results.to_csv(tables_output_path / "monte_carlo_results_summary.csv", index=False)
monte_carlo_plot_samples.to_csv(tables_output_path / "monte_carlo_plot_samples.csv", index=False)
monte_carlo_run_log.to_csv(tables_output_path / "monte_carlo_run_log.csv", index=False)

display(monte_carlo_results)


## Overload probability table


In [ ]:
overload_probability_table = monte_carlo_results[[
    "scenario",
    "margin_case",
    "distribution_model",
    "subsystem",
    "simulations",
    "capacity_kw",
    "mean_load_kw",
    "p95_load_kw",
    "p99_load_kw",
    "overload_count",
    "overload_probability",
    "overload_probability_pct",
    "rule_of_three_upper_if_zero",
]].copy()

overload_probability_table.to_csv(
    tables_output_path / "monte_carlo_overload_probability_table.csv",
    index=False,
)

display(overload_probability_table)


## Deterministic comparison check


In [ ]:
deterministic_comparison = monte_carlo_results[[
    "scenario",
    "margin_case",
    "distribution_model",
    "subsystem",
    "deterministic_load_kw",
    "mean_load_kw",
    "capacity_kw",
    "overload_probability_pct",
]].copy()

deterministic_comparison["mean_minus_deterministic_kw"] = (
    deterministic_comparison["mean_load_kw"] -
    deterministic_comparison["deterministic_load_kw"]
)
deterministic_comparison["mean_minus_deterministic_pct"] = (
    deterministic_comparison["mean_minus_deterministic_kw"] /
    deterministic_comparison["deterministic_load_kw"]
    * 100
)

deterministic_comparison.to_csv(
    tables_output_path / "monte_carlo_deterministic_comparison.csv",
    index=False,
)

display(deterministic_comparison)


## High-percentile capacity margins

No overloads were observed in the independent Monte Carlo simulation, so overload probability is not the most useful discriminator between subsystems. I therefore compare high-percentile spare capacity and P99 load-to-capacity ratio. These metrics show how close each subsystem gets to its capacity under uncertainty.


In [ ]:
capacity_margin_table = monte_carlo_results[[
    "scenario",
    "margin_case",
    "distribution_model",
    "subsystem",
    "capacity_kw",
    "p95_load_kw",
    "p99_load_kw",
    "p95_spare_capacity_kw",
    "p99_spare_capacity_kw",
    "p95_capacity_margin_pct",
    "p99_capacity_margin_pct",
    "p95_load_to_capacity_ratio",
    "p99_load_to_capacity_ratio",
]].copy()

capacity_margin_table.to_csv(
    tables_output_path / "monte_carlo_high_percentile_capacity_margins.csv",
    index=False,
)

display(capacity_margin_table)


## P99 load-to-capacity ratio

The P99 load-to-capacity ratio is useful because it puts HVAC, CW and FW onto the same scale. A value near 1 means the P99 sampled load is close to the available capacity.


In [ ]:
p99_ratio_table = monte_carlo_results[[
    "scenario",
    "margin_case",
    "distribution_model",
    "subsystem",
    "capacity_kw",
    "p99_load_kw",
    "p99_spare_capacity_kw",
    "p99_capacity_margin_pct",
    "p99_load_to_capacity_ratio",
]].copy()

p99_ratio_table = p99_ratio_table.sort_values(
    "p99_load_to_capacity_ratio",
    ascending=False,
).reset_index(drop=True)

p99_ratio_table.to_csv(
    tables_output_path / "monte_carlo_p99_load_to_capacity_ratio.csv",
    index=False,
)

display(p99_ratio_table)


## Plots


In [ ]:
if plotting_available:
    sns.set_theme(style="whitegrid")

    ratio_plot = monte_carlo_results.copy()
    ratio_plot["scenario_label"] = ratio_plot["scenario"].map({
        "Scenario Alpha": "Alpha",
        "Scenario Bravo": "Bravo",
    })
    ratio_plot["margin_case_label"] = ratio_plot["margin_case"].map({
        "Case A - CUM": "Case A",
        "Case B - CMM + CUM": "Case B",
        "Case C - CMM + CUM + DBM + IGM": "Case C",
    })

    graph = sns.catplot(
        data=ratio_plot,
        x="subsystem",
        y="p99_load_to_capacity_ratio",
        hue="distribution_model",
        row="margin_case_label",
        col="scenario_label",
        kind="bar",
        height=3.1,
        aspect=1.2,
        sharey=True,
    )
    graph.set_titles("{row_name} | {col_name}")
    graph.fig.suptitle("P99 Load-to-Capacity Ratio by Subsystem", y=1.03)
    graph.set_axis_labels("Subsystem", "P99 load / capacity")
    if graph._legend is not None:
        graph._legend.set_title("Uncertainty model")
    for axis in graph.axes.flat:
        axis.axhline(1.0, color="black", linewidth=1)
        axis.set_ylim(0, 1.05)
    graph.savefig(
        figures_output_path / "monte_carlo_p99_load_to_capacity_ratio.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

    spare_plot = monte_carlo_results.copy()
    spare_plot["scenario_label"] = spare_plot["scenario"].map({
        "Scenario Alpha": "Alpha",
        "Scenario Bravo": "Bravo",
    })
    spare_plot["margin_case_label"] = spare_plot["margin_case"].map({
        "Case A - CUM": "Case A",
        "Case B - CMM + CUM": "Case B",
        "Case C - CMM + CUM + DBM + IGM": "Case C",
    })

    graph = sns.catplot(
        data=spare_plot,
        x="subsystem",
        y="p99_spare_capacity_kw",
        hue="distribution_model",
        row="margin_case_label",
        col="scenario_label",
        kind="bar",
        height=3.1,
        aspect=1.2,
        sharey=False,
    )
    graph.set_titles("{row_name} | {col_name}")
    graph.fig.suptitle("P99 Spare Capacity by Subsystem", y=1.03)
    graph.set_axis_labels("Subsystem", "Capacity - P99 sampled load (kW)")
    if graph._legend is not None:
        graph._legend.set_title("Uncertainty model")
    for axis in graph.axes.flat:
        axis.axhline(0, color="black", linewidth=1)
    graph.savefig(
        figures_output_path / "monte_carlo_p99_spare_capacity.png",
        dpi=150,
        bbox_inches="tight",
    )
    graph.savefig(
        figures_output_path / "monte_carlo_high_percentile_spare_capacity.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

    graph = sns.displot(
        data=monte_carlo_plot_samples,
        x="sampled_load_kw",
        hue="distribution_model",
        row="scenario",
        col="subsystem",
        kind="hist",
        bins=40,
        common_bins=False,
        facet_kws={"sharex": False, "sharey": False},
        alpha=0.55,
        height=3.0,
        aspect=1.15,
    )
    graph.fig.suptitle("Sampled Cascaded Load Distributions", y=1.02)
    graph.set_axis_labels("Sampled cascaded load (kW)", "Count")
    if graph._legend is not None:
        graph._legend.set_title("Uncertainty model")
    graph.savefig(
        figures_output_path / "monte_carlo_sampled_load_distributions.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Skipping plots because matplotlib and seaborn are not installed in this Python environment.")


## Interpretation

The first risk ranking is based on P99 load-to-capacity ratio and P99 spare capacity. Because no overloads were observed, these high-percentile margin metrics are more informative than plotting overload probability alone.


In [ ]:
most_constrained = (
    monte_carlo_results
    .sort_values(
        ["p99_load_to_capacity_ratio", "p99_spare_capacity_kw"],
        ascending=[False, True],
    )
    .head(10)
    [[
        "scenario",
        "margin_case",
        "distribution_model",
        "subsystem",
        "capacity_kw",
        "p95_load_kw",
        "p99_load_kw",
        "p95_spare_capacity_kw",
        "p99_spare_capacity_kw",
        "p99_capacity_margin_pct",
        "p99_load_to_capacity_ratio",
        "overload_probability_pct",
        "rule_of_three_upper_if_zero",
    ]]
    .reset_index(drop=True)
)

most_constrained.to_csv(tables_output_path / "monte_carlo_most_constrained_cases.csv", index=False)
display(most_constrained)


## Notebook 05 conclusion

This notebook gives the first independent Monte Carlo estimate of overload risk for the cascaded cooling system. The important modelling choices are:

* local load rows are sampled directly;
* embedded workbook propagation rows remain excluded;
* the cascade is recomputed in every iteration;
* triangular and truncated normal uncertainty models are run side by side;
* all uncertainty cases are evaluated for both operating scenarios.

The next step is Notebook 06, where the independence assumption can be relaxed and dependency between sampled loads can be tested.
